<center><h1><mark>Funnel Drop-off Analysis & A/B Testing for Online Store</mark></h1></center>

<P>Performed funnel analysis on e-commerce user data to identify where users drop off in the purchase journey. Compared conversion rates across different categories and applied A/B testing to evaluate strategies for improving user conversion and overall sales.</P>

In [118]:
import pandas as pd
import os

In [120]:
df = pd.read_csv("2019-Oct.csv", nrows=300000)

In [121]:
print("\nShape of the dataset:")
print(df.shape)


Shape of the dataset:
(300000, 9)


In [122]:
print("\nEvent type distribution:")
print(df['event_type'].value_counts())


Event type distribution:
event_type
view        289638
purchase      5703
cart          4659
Name: count, dtype: int64


In [126]:
print("\nMissing values:")
print(df.isnull().sum())


Missing values:
event_time           0
event_type           0
product_id           0
category_id          0
category_code    95511
brand            42173
price                0
user_id              0
user_session         0
dtype: int64


In [128]:
print("\nFirst 5 rows:")
print(df.head())


First 5 rows:
                event_time event_type  product_id          category_id  \
0  2019-10-01 00:00:00 UTC       view    44600062  2103807459595387724   
1  2019-10-01 00:00:00 UTC       view     3900821  2053013552326770905   
2  2019-10-01 00:00:01 UTC       view    17200506  2053013559792632471   
3  2019-10-01 00:00:01 UTC       view     1307067  2053013558920217191   
4  2019-10-01 00:00:04 UTC       view     1004237  2053013555631882655   

                         category_code     brand    price    user_id  \
0                                  NaN  shiseido    35.79  541312140   
1  appliances.environment.water_heater      aqua    33.20  554748717   
2           furniture.living_room.sofa       NaN   543.10  519107250   
3                   computers.notebook    lenovo   251.74  550050854   
4               electronics.smartphone     apple  1081.98  535871217   

                           user_session  
0  72d76fde-8bb3-4e00-8c23-a032dfed738c  
1  9333dfbd-b87a-4708-9

<center><h1><mark>Funnel Analysis</mark></h1></center>

In [131]:
df['is_view'] = (df['event_type'] == 'view').astype(int)
df['is_cart'] = (df['event_type'] == 'cart').astype(int)
df['is_purchase'] = (df['event_type'] == 'purchase').astype(int)

In [133]:
user_funnel = df.groupby('user_id').agg({
    'is_view': 'max',
    'is_cart': 'max',
    'is_purchase': 'max'
}).reset_index()

user_funnel.head()

,user_id,is_view,is_cart,is_purchase
0,306441847,1,0,0
1,321655812,1,0,0
2,348609428,1,0,0
3,351866718,1,0,0
4,362699320,1,0,0


In [135]:
user_funnel['is_cart'] = user_funnel[['is_cart', 'is_purchase']].max(axis=1)

In [183]:
print(user_funnel['is_cart'].sum())
print(user_funnel['is_purchase'].sum())

5445
4433


In [137]:
total_users = user_funnel.shape[0]

view_users = user_funnel['is_view'].sum()
cart_users = user_funnel['is_cart'].sum()
purchase_users = user_funnel['is_purchase'].sum()

print("Total users:", total_users)
print("View users:", view_users)
print("Cart users:", cart_users)
print("Purchase users:", purchase_users)

Total users: 55882
View users: 55869
Cart users: 5445
Purchase users: 4433


In [139]:
view_to_cart = cart_users / view_users 
cart_to_purchase = purchase_users / cart_users
overall_conversion = purchase_users / view_users

print("View → Cart:", view_to_cart)
print("Cart → Purchase:", cart_to_purchase)
print("Overall Conversion:", overall_conversion)

View → Cart: 0.09746012994683993
Cart → Purchase: 0.8141414141414142
Overall Conversion: 0.07934632801732625


<h3><mark>There is a large drop between Viewing the product and add the product to Cart</mark></h3>

<h3>Let’s prove this using data</h3>

<p><mark>1. Certain product categories may have lower View → Cart conversion</mark></p>

In [144]:
df['category_code'] = df['category_code'].fillna('unknown')
category_conversion = df.groupby('category_code').agg({
    'is_view': 'sum',
    'is_cart': 'sum'
})

category_conversion['conversion_rate'] = category_conversion['is_cart'] / category_conversion['is_view']

category_conversion = category_conversion.sort_values(by='conversion_rate', ascending=False)

category_conversion.head(10)

,is_view,is_cart,conversion_rate
category_code,,,
electronics.smartphone,75591,3208,0.042439
electronics.audio.headphone,7320,269,0.036749
appliances.environment.water_heater,1397,43,0.030780
stationery.cartrige,50,1,0.020000
electronics.video.tv,6275,120,0.019124
auto.accessories.alarm,1575,29,0.018413
construction.tools.pump,347,6,0.017291
computers.peripherals.printer,1231,21,0.017059
construction.tools.saw,1471,24,0.016315


<p><mark>2. Lower-priced or unknown brand products may have lower conversion</mark></p>

<p><mark>3. Users may just be browsing, not intending to buy</mark></p>

<p>The largest drop-off occurs at the View to Cart stage, indicating that users are not sufficiently convinced to add products to their cart. However, once users reach the cart stage, the Cart to Purchase conversion is relatively high, suggesting strong purchase intent among those users. Therefore, the primary opportunity for improvement lies in increasing the View to Cart conversion, as it will have the highest impact on overall revenue.</p>

<h3><mark>Analyze conversion by product category</mark></h3>
<p>Because:</p>
<p>Maybe some categories perform well and some very poorly</p>

In [150]:

df['category_code'] = df['category_code'].fillna('unknown')

category_conversion = df.groupby('category_code').agg({
    'is_view': 'sum',
    'is_cart': 'sum'
})


In [152]:
category_conversion['conversion_rate'] = category_conversion['is_cart'] / category_conversion['is_view']

category_conversion = category_conversion.sort_values(by='conversion_rate', ascending=False)


In [154]:
print("Top categories:\n", category_conversion.head(5))
print("\nBottom categories:\n", category_conversion.tail(5))

Top categories:
                                      is_view  is_cart  conversion_rate
category_code                                                         
electronics.smartphone                 75591     3208         0.042439
electronics.audio.headphone             7320      269         0.036749
appliances.environment.water_heater     1397       43         0.030780
stationery.cartrige                       50        1         0.020000
electronics.video.tv                    6275      120         0.019124

Bottom categories:
                                    is_view  is_cart  conversion_rate
category_code                                                       
appliances.kitchen.hob                 686        0              0.0
appliances.kitchen.grill               549        0              0.0
appliances.kitchen.dishwasher          608        0              0.0
appliances.kitchen.coffee_machine      648        0              0.0
computers.components.memory            380        0

<p><mark>High conversion categories:</mark></p>
<p>smartphones → ~4.2%</p>
<p>headphones → ~3.6%</p>
<p><mark>Low conversion categories:</mark></p>
<p>kitchen appliances → 0%</p>
<p>computer components → 0%</p>

<p>Product category significantly affects conversion rates. Categories like smartphones and headphones show higher View to Cart conversion, indicating strong user demand and familiarity. In contrast, categories such as kitchen appliances and computer components show near-zero conversion, suggesting higher decision complexity or lower user intent. This indicates that improving product presentation and information in low-performing categories could help increase conversions.</p>

<center><h1><mark>A/B TESTING</mark></h1></center>

In [159]:
import numpy as np

user_funnel['variant'] = np.random.choice(['A', 'B'], size=len(user_funnel))

<p>A → Random group of users</p>
<p>B → Another random group of users</p>

In [162]:
ab_test = user_funnel.groupby('variant')['is_purchase'].mean()

print(ab_test)

variant
A    0.079394
B    0.079260
Name: is_purchase, dtype: float64


<p>A → 8.09%</p>
<p>B → 7.77%</p>
<p>Difference = ~0.32%</p>

In [169]:
ab_counts = user_funnel.groupby('variant')['is_purchase'].agg(['sum', 'count'])
print(ab_counts)

          sum  count
variant             
A        2239  28201
B        2194  27681


In [171]:
from statsmodels.stats.proportion import proportions_ztest
success = ab_counts['sum'].values
total = ab_counts['count'].values
z_stat, p_value = proportions_ztest(success, total)
print("Z-stat:", z_stat)
print("P-value:", p_value)

Z-stat: 0.05869373018413118
P-value: 0.9531960533186261


<p>p-value > 0.05 --> The difference is NOT statistically significant</p>
<p>Although variant A shows a slightly higher conversion rate than variant B, the difference is not statistically significant (p > 0.05). Therefore, we cannot conclude that one variant performs better than the other, and the observed difference is likely due to random variation.</p>

<h1><mark>Conclusion:</mark></h1>
<p>Focus on improving the View to Cart conversion by optimizing low-performing product categories such as kitchen appliances and computer components. This can be done by enhancing product descriptions, pricing strategies, and user experience. Since high-performing categories like smartphones already show strong conversion, improving weaker categories can significantly increase overall conversion rates.</p>

In [181]:
user_funnel.to_csv("user_funnel.csv", index=False)